## Task to solve
**Business Task:** 
Build an ML model that predicts future occupancy/demand for each listing and gives hosts recommendations on how to increase bookings

**Business Problem**
Airbnb hosts and property managers want to maximize their revenue and occupancy but struggle to understand which factors truly drive booking success and high guest satisfaction. Many listings underperform despite good locations, while others outperform expectations.
Goal: Build an ML system that predicts the future success of a listing and provides actionable recommendations to hosts on how to improve performance.

**Target variable** = Occupancy Rate next 30 days

**Key Features You Can Use**
**Host-related:**

host_response_time, host_response_rate, host_acceptance_rate
host_is_superhost, host_identity_verified
calculated_host_listings_count (multi-listing hosts vs single)
host_since (experience)

**Property & Listing:**

property_type, room_type
accommodates, bedrooms, bathrooms_text
amenities (needs parsing)
instant_bookable

**Location & Market:**

neighbourhood_cleansed, latitude, longitude
neighbourhood_group_cleansed (if available in other cities)

**Listing Quality:**

Length and sentiment of description + neighborhood_overview
name length/sentiment (NLP features)

**Availability & History:**

availability_30/60/90/365
minimum_nights, maximum_nights

## Different ways
**Market Segmentation / Clustering**

* Task: Group similar listings into segments (budget, luxury, family, business, unique experiences, etc.).
* Type: Unsupervised (K-Means, DBSCAN, Hierarchical, or embedding-based clustering).
* Features: Property type, amenities (parsed), location, price behavior, occupancy patterns.
* Value: Targeted marketing, personalized recommendations, competitive analysis.

**Superhost / High-Performer Prediction**

* Task: Predict which hosts/listings will achieve Superhost status or high long-term success.
* Target: host_is_superhost or a custom "High Performer" label (based on reviews + occupancy).
* Type: Classification (or Ranking).
* Value: Help new hosts understand what changes they need to reach top status.

**Listing Churn / Deactivation Prediction**

* Task: Predict which listings are likely to be removed or become inactive in the next 3-6 months.
* Target: Create label from availability table (e.g., "inactive if no bookings in 60+ days").
* Type: Binary classification or Survival Analysis.
* Value: Platform can intervene early with support for at-risk hosts.

## Data preparation

### Listing

#### Грузим данные

In [1]:
import pandas as pd

listing_path = "initial_data/listings.csv.gz"
listing_df = pd.read_csv(listing_path)

In [9]:
listing_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 81853 entries, 0 to 81852
Data columns (total 79 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            81853 non-null  int64  
 1   listing_url                                   81853 non-null  str    
 2   scrape_id                                     81853 non-null  int64  
 3   last_scraped                                  81853 non-null  str    
 4   source                                        81853 non-null  str    
 5   name                                          81853 non-null  str    
 6   description                                   79140 non-null  str    
 7   neighborhood_overview                         39600 non-null  str    
 8   picture_url                                   81851 non-null  str    
 9   host_id                                       81853 non-null  int64  
 1

#### Удаляем ненужные столбцы

In [178]:
listing_df2 = listing_df.copy()

# Определяем список колонок к удалению
drop_list = []
for col in listing_df.columns:

    # Добавляем в список полностью пустые колонки
    if listing_df[col].isnull().mean() == 1:
        drop_list += [col]
        continue

    # Добавляем в список колонки с одним значением
    if listing_df[col].nunique() == 1:
        drop_list += [col]
        continue

    # Добавляем в список колонки с ссылками
    if 'url' in col:
        drop_list += [col]
        continue

# Дополнительно убираем поля
drop_list += ['host_name',  # Имя хоста. Бесполензо для модели
              'estimated_occupancy_l365d',  # Cлишком сильно может коррелировать с нашей метрикой
              'host_id',  # id хоста
              'availability_60',  # То же самое, что и таргет в 30 дн. Сильно коррелирует с таргетом
                'availability_90',  # То же самое, что и таргет в 30 дн. Сильно коррелирует с таргетом
                'availability_365',  # То же самое, что и таргет в 30 дн. Сильно коррелирует с таргетом
                'availability_eoy',  # То же самое, что и таргет в 30 дн. Сильно коррелирует с таргетом
                'id',  # id объекта
                'calendar_last_scraped',  # дата скржпинга
            ]

listing_df2.drop(columns=drop_list, inplace=True)

# Предобразуем таргет
listing_df2['availability_30'] = listing_df2['availability_30'] / 30

#### Кодируем время ответа

In [179]:
# Кодируем время ответа
responce_time_coding = {
    'within an hour': 1,
    'within a few hours': 3,
    'within a day': 24,
    'a few days or more': 72,
}
listing_df2['host_response_time'] = listing_df2['host_response_time'].replace(responce_time_coding)

# Заполняем незаполнные значения максимальной категорией
listing_df2['host_response_time'] = listing_df2['host_response_time'].fillna(72)  


#### Обратабывает отвечаемость и подтверждаемость хоста

In [180]:

for col in ['host_response_rate', 'host_acceptance_rate']:
    # Переводим из % в float
    listing_df2[col] = listing_df2[col].str.replace('%', '').apply(float) / 100

    # Заполняем пропуски 0
    listing_df2[col] = listing_df2[col].fillna(0)


#### Способы верификации хоста

In [181]:
import ast 

# Преобразуем строку в реальный список py
listing_df2['host_verifications'] = listing_df2['host_verifications'].apply(lambda x: ast.literal_eval(x) if not pd.isna(x) else None)

# Раскрываем список, делаем энкодинг и схлопываем до изначального размера
dummies_df = pd.get_dummies(listing_df2['host_verifications'].explode(), prefix='host_verifications').groupby(level=0).max()

# Объединяем с исходным df
listing_df2 = pd.concat([listing_df2, dummies_df], axis=1)


#### Длительность на платформе

In [182]:
# Преобразуем в даты
listing_df2['host_since'] = pd.to_datetime(listing_df2['host_since'])
listing_df2['last_scraped'] = pd.to_datetime(listing_df2['last_scraped'])

# Считаем возраст хоста на платформе
listing_df2['host_age_on_platform'] = (listing_df2['last_scraped'] - listing_df2['host_since']).dt.days / 365

listing_df2['host_age_on_platform'] = listing_df2['host_age_on_platform'].fillna(0)

# Дропаем ненужное
listing_df2.drop(columns=['host_since', 'last_scraped'], inplace=True)

#### Форматируем булевые колонки

In [183]:
bool_columns_list = [
    'host_is_superhost',
    'host_has_profile_pic',
    'host_identity_verified',
    'instant_bookable',
]

for col in bool_columns_list:

    listing_df2[col] = listing_df2[col].replace({'f': False, 't': True})

    listing_df2[col] = listing_df2[col].fillna(False)


### Считаем таргет

In [184]:
# # calendar_path = "initial_data/calendar.csv.gz"
# calendar_df = pd.read_csv(calendar_path)

In [ ]:
# calendar_df['date'] = pd.to_datetime(calendar_df['date'])

In [ ]:
# from datetime import timedelta

# target_df = pd.merge(
#     left=listing_df2[['id', 'calendar_last_scraped']],
#     right=calendar_df,
#     left_on='id',
#     right_on='listing_id',
#     how='left',
# )

# target_df['calendar_last_scraped'] = pd.to_datetime(target_df['calendar_last_scraped'])

# # Определяем правую границу интервала дат
# target_df['date_to'] = target_df['calendar_last_scraped'] + timedelta(days=30)

# # Оставлем тлк данные за  первые 30 дней после даты скрэпинга и только занаятые дни
# target_df = target_df[
#         (target_df['date'] >= target_df['calendar_last_scraped']) &
#         (target_df['date'] < target_df['date_to']) &
#         (target_df['available'] == 'f')  
#     ]

# # Считаем кол-во дней заняости в ближайшие 30 дней
# target_df = target_df.groupby(by='id', as_index=False)['date'].nunique()

# # Считаем занятость в ближайшие 30 дней
# target_df['date'] = target_df['date'] / 30

# # Переименовываем
# target_df.rename(columns={'date': 'occupancy_30d'}, inplace=True)

#### Джоиним к основной витрине с фичами

In [ ]:
# listing_df3 = pd.merge(
#     left=listing_df2,
#     right=target_df,
#     on='id',
#     how='left',
# )

# # Если ничего не заджоинилось, значит объект полностью свободен
# listing_df3['occupancy_30d'] = listing_df3['occupancy_30d'].fillna(0)

In [187]:
listing_df2.to_pickle(r'D:\AV\Education\Master\ML\Project\prepared_data\data.pkl')